**This code present prompts stratgies for Covid-19 Vaccine.**
Since the first approach not adopt, we adopt only the two latter approaches that used in feminism movement.

In [ ]:
pip install openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 2.7 MB/s eta 0:00:00


In [ ]:
import openai
import csv
import random

In [ ]:
openai.api_key='OPENAI_API_KEY'

##Second approach: Specific prompt for each stance

In [ ]:
def create_tweets(prompt_, total_tweet):

    prompt = prompt_
    Generated_tweets = []
    #ask ChatGPT create tweets
    response = openai.ChatCompletion.create(
    model="gpt-3.5-turbo",
    messages=[
            {"role": "system", "content": "You are a tweet generator"},
            {"role": "user", "content": prompt}
        ],
    max_tokens=100,
    n=total_tweet,
    temperature=1
    )

    #add the responses to the list
    for respon in response['choices']:
      tweet = respon['message']['content'].strip()
      Generated_tweets.append(tweet)

    return Generated_tweets

#save the tweets in csv file
def save_tweets_csv(tweets, label):
    with open('covid19_inst_new.csv', 'w', newline='', encoding='utf-8-sig') as csvfile:
        colum_names = ['Tweet', 'Label']
        writer = csv.DictWriter(csvfile, fieldnames= colum_names)
        writer.writeheader()

        for tweet in tweets:
            writer.writerow({'Tweet': tweet, 'Label': label})

In [ ]:
#prompts for each stance
#for each set of tweets we slightly chang the Task and use different tweets examples ...

#1: for favour satnce

prompt1 = '''
    Task: create an informal tweet that support Covid-19 vaccine. The tweet may include mention and hashtag.
    Type of Text: Short Tweet (Maximum of 30 words).
    The output: Should include the tweet ONLY.

    Example: "@DonaldJTrumpJr @realDonaldTrump @VP @IvankaTrump  Good news.  Will take Covid-19 vaccine first to ease out safety concerns: Pfizer CEO"
    Example: "Again, NOBODY is explaining that, in the vaccine world, 90% is like 100%.  Because of herd immunity.  We DON’T NEED it to be 100% effective.  Because of probabilities. URL"
    Example: "Ok so in honor of the good news about the vaccines so far that’s been on the news, would you get the covid-19 vaccine?"
    '''
#2: for Against
prompt2 = '''
    Task: create an informal tweet that goes against Covid-19 vaccine. The tweet may include mention and hashtag.
    Type of Text: Short Tweet (Maximum of 30 words).
    The output: Should include the tweet ONLY.

    Example: "Do not take ANY vaccine before 5 years testing"
    Example: "@realDonaldTrump No need for vaccine with 90% effective, we need 100% effective. Coronavirus killing less than 3% of the infected people ."
    Example: "#BBC: Has this type of vaccine ever been used before?  There are no RNA vaccines that have been approved for use in people. "

    '''
#3: for NONE stance
prompt3 = '''
    Task: create tweet contain neutral news about Covid-19 vaccine. The tweet may include mention and hashtag.
    Type of Text: Short Tweet (Maximum of 30 words).
    The output: Should include the tweet ONLY.

    Example: " Brazil allows trials of Chinese Covid-19 vaccine to resume"
    Example: "FOX NEWS: Coronavirus vaccine should go to health care workers, long term care facilities first, CDC panel recommends"
    Example: "Pfizer CEO says COVID-19 vaccine candidate has passed FDA safety milestone "

    '''


In [ ]:
# create tweets by choose the stance prompt 1, 2, or 3 and specify number of tweets
total_tweet= 100 #(no more than 128 as it limited number for n parameter)
label='FAVOR'
synthetic_tweets = create_tweets(prompt1, total_tweet)

# save tweets in CSV
save_tweets_csv(synthetic_tweets, label)

# Third approach: Template prompt for each stance



In [ ]:
#function to ask ChatGPT
def create_tweets(sys_prompt, prompt):

    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": prompt},
        ],
        max_tokens=100,
        n=1,
        temperature=1,
    )
    return response.choices[0].message.content.strip()

#function to set the hint sentence and then call create_tweets to create the tweets
def set_propmts(num_tweets, templ, hasht, topic, base_P):
    tweets = []
    templates = templ
    hashtags = hasht
    topics = topic

    for _ in range(num_tweets):
        template = random.choice(templates)
        hashtag = random.choice(hashtags)
        topic = random.choice(topics)
        prompt = f"{template} {topic}. {hashtag}"
        base_prompt= base_P
        full_prompt= "Hint: "+ prompt
        tweet = create_tweets(base_prompt, full_prompt)
        tweets.append(tweet)

    return tweets


In [ ]:
#1: prompts for Favor stance consist of base_prompt1 for the system, and the hint sentence lists' to create hint sentence.
#Important: make sure the lists coherent each for each subset of tweets by add comments to some elements to avoid unwanted results.
base_prompt1='''
    You are a tweet generator.
    Task: create an informal tweet that support Covid-19 vaccine. Inspire your perspective from the hint.
          The hint is consist of "template, Topic. hashtag", use your creativity to generate the tweet without copy the template.
    Type of Text: Short Tweet (Maximum of 30 words) and include the tweet only.

    Example: "Big news... #COVID19 ..", [Pfizer-BioNTech vaccin]. #CovidVaccine"
    Answer: "OMG, you won't believe this! 😱 The Pfizer-BioNTech vaccine is like a superhero in the battle against #COVID19! Time to unleash our inner heroes and get vaxxed! Let's save the world, one shot at a time! 🌎💉 #VaxHeroes"

    Example: "a significant number of ... say", "[vaccine data improve its benefits]". #Covid19"
    Answer: "Hey fam, listen up! 🗣️ The vaccine data is getting a major thumbs up from experts, and that's some good stuff! It's time to roll up our sleeves and give #Covid19 a knockout punch! Let's do this, together! 🥊💉 #VaccineWin"

    Example: "a significant number of ... say", "[the need for back to jobs]". #vaccine"
    Answer: "You heard it right! The experts have spoken, and the vaccine is a game-changer! Let's reclaim our jobs and kick COVID to the curb! #JobRevival #VaccinePower"
'''
templates1 = [
      #"Big news... #COVID19 ..",
      "Congratulations to the ... team! ...",
     "a significant number of ... say",
      "BREAKING‼️Updated results from...",
     # "... fingers crossed!...",
      "OK!!!! US Presidents ...",
      "make safe and effective #vaccines available to everyone...",
      "@user:  Naturally acquired COVID-19 vaccine"
       ]
hashtags1=[
    "#vaccine",
    "#COVID19",
    "#CovidVaccine"
]

topics1 = [
     "[Pfizer-BioNTech vaccine]",
     "[potential impact of taking the vaccine]",
     #"[Encouragement getting vaccinated]",
     "[the need for back to jobs]",
    # "[Optimism future with Pfizer vaccine]",
     "[vaccine data improve its benefits]",
     "[Positive news about the vaccine]"
    ]

#2: prompts for Against stance
base_prompt2='''
    You are a tweet generator.
    Task: create an informal tweet that goes against Covid-19 vaccine. Inspire your perspective from the hint.
          The hint is consist of "template, Topic. hashtag", use your creativity to generate the tweet without copy the template.
    Type of Text: Short Tweet (Maximum of 30 words) and include the tweet only.

    Example: "Yup no thank you!!!...", "[the vaccine may alter the DNA]". #Covid19"
    Answer: "People acting like the COVID-19 vaccine is some kind of miracle cure. Nah, I'll pass! 😅 My body, my choice, my immune system! #Vaccine"

    Example: ""I don't care! ...", "[vaccine side effects]". #vaccine"
    Answer: "The pressure to get vaccinated is real, but I'm standing my ground. No vaccine for me! #StayTrue"

    Example: "@CovidData2 Remember this ...", "[vaccine side effects]". #covid19"
    Answer: "Just a friendly reminder that vaccines can have some pretty nasty side effects. No thanks, I'll take my chances without one!"
'''
templates2 = [
    #"Yup no thank you!!!...",
    "I don't care! ...",
    "Bylsma Facebook comment that: ...",
  #  "Not yet ... they still have",
   # "@CovidData2 Remember this ...",
    "Please sign and SHARE this link -",
 #   "Covid-19 is a hoax...",
    "Call me crazy but I’m not gonn.."

]
hashtags2=[
    "#vaccine",
    "#COVID19",
    "#CovidVaccine"
]
topics2 = [
    "[Not trust in the vaccine]",
   # "[skepticism about the effectiveness of the vaccine]",
    "[vaccine may altere the DNA]",
    #"[skepticism about vaccine's development]",
    "[vaccine side effects]",
    "[vaccine development is for financial purpose]"
]

#3: prompts for None stance
base_prompt3='''
    You are a tweet generator.
    Task: create an informal tweet that not have clear stance (netural) or not related to covid-19 vaccine. Inspire your perspective from the hint. use the exact hashtag.
          The hint is consist of "template, Topic. hashtag", use your creativity to generate the tweet without copy the template.
    Type of Text: Short Tweet (Maximum of 30 words) and include the tweet only.

    Example: "Pfizer CEO says ...", "[vaccine availability and health implications]". #Covid19"
    Answer: "In a recent statement, Pfizer's CEO discusses the progress made in vaccine development and distribution, highlighting the challenges ahead. #COVID19 #VaccineUpdate"

    Example: "Health care workers: ....", "[ vaccine trials in the UK]". #vaccine"
    Answer: Health care workers play a pivotal role in COVID-19 vaccine administration, ensuring the vaccination drive remains efficient and effective. #HealthcareHeroes #COVIDVaccine"

    Example: "FOX NEWS: Coronavirus vaccine should ..", "[vaccine distribution and doses among contries]". #vaccine
    Answer: FOX NEWS reports on the ongoing discussions about COVID-19 vaccine distribution and the need for equitable access across countries.
'''
templates3 = [
       "Update news on a coronavirus vaccine ...",
       "Health care workers: ....",
       "Pfizer CEO says ...",
       "FOX NEWS: Coronavirus vaccine should ..",
       "The governments announced: "
]
hashtags3=[
    "#vaccine",
    "#COVID19",
    "#CovidVaccine"

]
topics3= [
    #"[ vaccine trials in the UK]",
    "[data-related discussions about vaccine]",
    "[News and updates about vaccine in UK]",
    "[vaccine availability and health implications]",
   #"[Information about coronavirus in different regions]",
    "[vaccine distribution and doses among contries]"
]

In [ ]:
#call the function for each stance separately
num_tweets = 50   #set number of tweets to create subset of tweets by choos the elements of the lists
feminist_favour = set_propmts(num_tweets, templates1, hashtags1, topics1, base_prompt1 )
#feminist_against = set_propmts(num_tweets, templates2, hashtags2, topics2, base_prompt2 )
#feminist_none = set_propmts(num_tweets, templates3, hashtags3, topics3, base_prompt3 )

#save the tweets into CSV file
with open("covid19_template_fav.csv", 'w', newline='', encoding='utf-8-sig') as file:
    writer = csv.writer(file)
    writer.writerow(["Tweet", "Label"])
    for data in feminist_favour:
        writer.writerow([data, "FAVOR"])